# Ticketing-Chatbot — GPU training on Colab

Fine-tunes `bert-base-multilingual-cased` on the 20k-ticket dataset and exports the
**inference bundle** (model + tokenizer + label map + config) plus evaluation reports.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
Expect roughly 15–25 minutes per epoch on a T4; early stopping usually ends the run early.

**After the last cell:** a `ticketing_bundle.zip` download starts automatically.
Unzip it into the repo root on your machine so `models/bundle/` and `reports/` are populated.

In [ ]:
# Verify a GPU is attached (should print a Tesla T4 or similar)
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
!git clone --depth 1 https://github.com/KR-16/Ticketing-Chatbot.git
%cd Ticketing-Chatbot
!pip install -q -r requirements.txt

In [ ]:
# Run the full pipeline: load -> preprocess -> train -> evaluate -> save bundle.
# MLflow falls back to local file tracking automatically (no server on Colab).
!python main.py

In [ ]:
# Display the evaluation results produced by the run
import json
from IPython.display import Image, display

with open("reports/metrics.json") as f:
    metrics = json.load(f)

print(f"Accuracy : {metrics['accuracy']:.4f}")
print(f"Macro F1 : {metrics['macro_f1']:.4f}\n")
for cls in ["Change", "Incident", "Problem", "Request"]:
    row = metrics["per_class"][cls]
    print(f"{cls:<10} precision={row['precision']:.3f} recall={row['recall']:.3f} f1={row['f1-score']:.3f}")

display(Image("reports/confusion_matrix.png"))

In [ ]:
# Zip the inference bundle + reports and download to your machine
!zip -r -q ticketing_bundle.zip models/bundle reports
!ls -lh ticketing_bundle.zip
from google.colab import files
files.download("ticketing_bundle.zip")

## Next steps

1. Unzip `ticketing_bundle.zip` into the repo root locally — you should end up with
   `models/bundle/` (used by the predictor, API, and demo) and `reports/`.
2. Commit `reports/metrics.json` + `reports/confusion_matrix.png` so the README
   results table can cite them (`models/` stays gitignored — the bundle is a build
   artifact, not source).
3. Copy accuracy / macro-F1 into the README results table.